[![Fixel Algorithms](https://fixelalgorithms.co/images/CCExt.png)](https://fixelalgorithms.gitlab.io)

# Deep Learning Methods

## Deep Learning - Vision and Language - Image & Text Embedding

This notebooks shows how to use Image and Text Embedding model for _Zero Shot Image Classification_.

> Notebook by:
> - Royi Avital RoyiAvital@fixelalgorithms.com

## Revision History

| Version | Date       | User        |Content / Changes                                                   |
|---------|------------|-------------|--------------------------------------------------------------------|
| 1.0.000 | 13/10/2026 | Royi Avital | First version                                                      |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FixelAlgorithmsTeam/FixelCourses/blob/master/AIProgram/2026_02/0130DeepLearningDiffusion.ipynb)

In [ ]:
# Import Packages

# General Tools
import numpy as np
import scipy as sp
import pandas as pd

# Machine Learning
from sklearn.metrics import accuracy_score, f1_score
from umap import UMAP

# Image Processing
import imageio.v3 as iio

# Deep Learning
import torch
import os
os.environ['HF_HUB_DISABLE_XET'] = '1'
from transformers import AutoModel, AutoProcessor, AutoTokenizer

# Miscellaneous
import json
import os
import random

# Typing
from typing import Callable, Dict, Literal, Optional, Tuple, Union
from numpy.typing import NDArray

# Visualization
import matplotlib.pyplot as plt

# Jupyter
from IPython import get_ipython
from IPython.display import JSON, Markdown, display

## Notations

* <font color='red'>(**?**)</font> Question to answer interactively.
* <font color='blue'>(**!**)</font> Simple task to add code for the notebook.
* <font color='green'>(**@**)</font> Optional / Extra self practice.
* <font color='brown'>(**#**)</font> Note / Useful resource / Food for thought.

Code Notations:

```python
someVar    = 2; #<! Notation for a variable
vVector    = np.random.rand(4) #<! Notation for 1D array
mMatrix    = np.random.rand(4, 3) #<! Notation for 2D array
tTensor    = np.random.rand(4, 3, 2, 3) #<! Notation for nD array (Tensor)
tuTuple    = (1, 2, 3) #<! Notation for a tuple
lList      = [1, 2, 3] #<! Notation for a list
dDict      = {1: 3, 2: 2, 3: 1} #<! Notation for a dictionary
oObj       = MyClass() #<! Notation for an object
dfData     = pd.DataFrame() #<! Notation for a data frame
dsData     = pd.Series() #<! Notation for a series
hObj       = plt.Axes() #<! Notation for an object / handler / function handler
```

### Code Exercise

 - Single line fill

```python
valToFill = ???
```

 - Multi Line to Fill (At least one)

```python
# You need to start writing
?????
```

 - Section to Fill

```python
#===========================Fill This===========================#
# 1. Explanation about what to do.
# !! Remarks to follow / take under consideration.
mX = ???

?????
#===============================================================#
```

In [ ]:
# Configuration
# %matplotlib inline

seedNum = 512
np.random.seed(seedNum)
random.seed(seedNum)

# Matplotlib default color palette
lMatPltLibclr = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']
# sns.set_theme() #>! Apply SeaBorn theme

runInGoogleColab = 'google.colab' in str(get_ipython())

In [ ]:
# Constants

FIG_SIZE_DEF    = (8, 8)
ELM_SIZE_DEF    = 50
CLASS_COLOR     = ('b', 'r')
EDGE_COLOR      = 'k'
MARKER_SIZE_DEF = 10
LINE_WIDTH_DEF  = 2

PROJECT_NAME       = 'FixelCourses'
DATA_FOLDER_NAME   = 'DataSets'
MODELS_FOLDER_NAME = 'Models'
BASE_FOLDER_PATH   = os.getcwd()[:(len(os.getcwd()) - (os.getcwd()[::-1].lower().find(PROJECT_NAME.lower()[::-1])))]
DATA_FOLDER_PATH   = os.path.join(BASE_FOLDER_PATH, DATA_FOLDER_NAME)
MODELS_FOLDER_PATH = os.path.join(BASE_FOLDER_PATH, MODELS_FOLDER_NAME)

In [ ]:
# Download Auxiliary Modules for Google Colab
if runInGoogleColab:
    !wget https://raw.githubusercontent.com/FixelAlgorithmsTeam/FixelCourses/master/AIProgram/2024_02/DataManipulation.py
    !wget https://raw.githubusercontent.com/FixelAlgorithmsTeam/FixelCourses/master/AIProgram/2024_02/DataVisualization.py
    !wget https://raw.githubusercontent.com/FixelAlgorithmsTeam/FixelCourses/master/AIProgram/2024_02/DeepLearningPyTorch.py

In [ ]:
# Courses Packages

from DataVisualization import PlotConfusionMatrix, PlotLabelsHistogram

In [ ]:
# General Auxiliary Functions

def ParseEnvFile( fileName: str = '.env', *, filePath: str = '.', keyValSep: str = '=' ) -> Dict[str, str]:
    # Read the file and parse it into a dictionary
    dEnv = {}
    with open(os.path.join(filePath, fileName), 'r') as hF:
        for line in hF:
            if line.startswith('#') or not line.strip():
                continue
            key, value = line.strip().split(keyValSep, 1)
            dEnv[key]  = value.strip()
    return dEnv

## Motivation

Let a dataset be composed of 10 _classes_ with 15 images per class.  
The task is to classify the images.

<!-- ![](https://i.imgur.com/DCe552s.jpeg) -->
![](https://i.postimg.cc/YCWMLrRp/image.png)

How can the problem be solved?  
Can it be solved **without** a specific training?

## Multi Modal Encoders / Multi Modal Embedding Models

A _Modality_ is a type of data: _Text_, _Image_, _Audio_ or _Video_.  
An _Encoder_ converts an input into a vector of numbers.  
This vector is called an _Embedding_. It represents learned features of the input.

An _Image Encoder_ maps an image to an embedding.  
A _Text Encoder_ maps a chunk of text (Word / Sentence / Passage) to an embedding.

### A Shared Embedding Space

In order to generate a shared embedding space 2 constraints must be forced:

 - The embedding space of the text and images must match in their _dimension_.
 - The representations must be _aligned_.

Related images and text should have similar embeddings.  
During training, matching image and text pairs are encouraged to have higher similarity.  
Mismatched pairs are encouraged to have lower similarity.  
The encoders learn a _Joint / Shared Embedding Space_.

* <font color='brown'>(**#**)</font> A well known example is the [OpenAI CLIP Model](https://openai.com/index/clip).  
  Its name stands for [Contrastive Language Image Pre Training](https://en.wikipedia.org/wiki/Contrastive_Language%E2%80%93Image_Pre-training).  
* <font color='brown'>(**#**)</font> This notebook covers the evolution of `CLIP` (2021) -> `SigLIP` (2023) -> `SigLIP 2` (2025).

## The Contrastive Language Image Pre Training (CLIP) Model

CLIP learns from matching _Image / Text_ pairs.  
An image encoder processes the images. A text encoder processes their descriptions.  
The task during training is to find which image matches which text.

![](https://i.imgur.com/LjZfI95.png)
<!-- ![](https://i.postimg.cc/8CVhvChq/image.png) -->

### Training: Compare Every Pair

The training is based on having a dataset of Image and Text pairs.  
For instance, an image of a sunflower and the text _"a photo of a sunflower"_.  

The training is driving the model to learn a _similarity matrix_:

 - Take a batch of 4 matching pairs.  
 - Compare **every image** with **every text**.  
   This gives a $4 \times 4$ _Similarity Matrix_.  
 - The loss function maximizes the value on diagonal and minimizes the values off diagonal.

The comparison is based on _Cosine Similarity_.

The following table illustrates an example:

| Image / Text     | "an airplane"    | "a helicopter"   | "a sunflower"   | "a pizza"       |
|------------------|------------------|------------------|-----------------|-----------------|
| Airplane Image   | **0.82**         | 0.43             | 0.09            | 0.05            |
| Helicopter Image | 0.38             | **0.79**         | 0.07            | 0.04            |
| Sunflower Image  | 0.06             | 0.08             | **0.85**        | 0.12            |
| Pizza Image      | 0.03             | 0.05             | 0.14            | **0.88**        |

The loss function drives teh model to learn such matrices per batch.

### Learn in Both Directions

Per batch the loss is composed of 2 directional losses:  

 - Image to Text: 
    - Pick an image.
    - Calculate similarity to each text.
    - Scale the result by a learned parameter.
    - Apply classification loss with the _Cross Entropy Loss_.
 - Text to Image: 
    - Pick a text.
    - Calculate similarity to each image.
    - Scale the result by a learned parameter.
    - Apply classification loss with the _Cross Entropy Loss_.

The loss is a weighted sum of both directions.

* <font color='red'>(**?**)</font> Why might an airplane image have some similarity to the text _"a helicopter"_?
* <font color='brown'>(**#**)</font> One may consider a loss vs. the Identity matrix.  
  Yet it will force perpendicularity off diagonal which is a strong geometrical constraint and might be wrong contextually (See question above).
* <font color='brown'>(**#**)</font> The table contains similarities, not probabilities.  
  Softmax probabilities depend on the other candidates and the score scaling.
* <font color='brown'>(**#**)</font> Different pairs in a batch may still describe similar content.  
  Treating all off diagonal pairs as mismatches is a training assumption and a weakness of the model.

In [ ]:
# Parameters

# Data
dataDirName = 'ImgZeroShotCls' #<! https://huggingface.co/datasets/Royi/DataSets -> `ImgZeroShotCls.zip`
lImgFileExt = ['.jpg', '.png']

# Model
modelId   = "google/siglip2-base-patch16-224" #<! https://huggingface.co/google/siglip2-base-patch16-224
runDevice = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu') #<! The 1st CUDA device

## The Task: Classify Images into 10 Classes with No Training

<!-- ![](https://i.imgur.com/DCe552s.jpeg) -->
![](https://i.postimg.cc/YCWMLrRp/image.png)

* <font color='red'>(**?**)</font> Given a _CLIP_ like model, how would you solve the problem above?

## Generate / Load Data

 - Download the file `ImgZeroShotCls.zip` from https://huggingface.co/datasets/Royi/DataSets.
 - Unpack it into `FixelCourses\DataSets`.
 - Verify images in `FixelCourses\DataSets\ImgZeroShotCls`.

In [ ]:
# Generate / Load Data

dataDirPath = os.path.join(DATA_FOLDER_PATH, dataDirName)

# The folder is organized:
#  - ImgZeroShotCls
#     - <Class Name 001>
#       - <Image Files>
#     - <Class Name 002>
#       - <Image Files>

dData = {}

for clsName in sorted(os.listdir(dataDirPath)):
    clsDirPath = os.path.join(dataDirPath, clsName)
    if not os.path.isdir(clsDirPath):
        continue

    for imgFileName in sorted(os.listdir(clsDirPath)):
        imgFilePath = os.path.join(clsDirPath, imgFileName)
        if os.path.isfile(imgFilePath) and os.path.splitext(imgFileName)[1].lower() in lImgFileExt:
            dData[os.path.join(clsName, imgFileName)] = clsName

lImgFile   = list(dData)
lClsName   = sorted(set(dData.values()))
numSamples = len(dData)
numClasses = len(lClsName)

print(f'Number of images: {numSamples}')
print(f'Number of classes: {numClasses}')

### Plot the Data

In [ ]:
# Plot the Data

hF, vHa = plt.subplots(nrows = 2, ncols = 5, figsize = (10, 4))
vHa = vHa.flat

for clsName, hA in zip(lClsName, vHa):
    lClsImgFile = [imgFileName for imgFileName in lImgFile if dData[imgFileName] == clsName]
    imgFileName = random.choice(lClsImgFile)
    imgFilePath = os.path.join(dataDirPath, imgFileName)

    tI = iio.imread(imgFilePath)

    hA.imshow(tI)
    hA.set_title(clsName)
    hA.axis('off')

hF.tight_layout()

In [ ]:
# Plot the Class Labels Histogram

vY = np.array([lClsName.index(dData[imgFileName]) for imgFileName in lImgFile])

hF, hA = plt.subplots(figsize = (8, 5))
hA = PlotLabelsHistogram(vY, hA = hA, lClass = lClsName, xLabelRot = 45)
hF.tight_layout()

* <font color='red'>(**?**)</font> Is the whole image needed in order to accomplish the task? Think about optimizing the cost.

## The Hugging Face Transformers Package

[Hugging Face Transformers](https://huggingface.co/docs/transformers) is an open source Python package.  
It provides implementations of many models for _Text_, _Images_, _Audio_ and combinations of them.  
Its main contribution is a **common interface** for loading, using and fine tuning _Pre Trained Models_.

Instead of implementing each architecture ourselves, we can reuse existing models and learned weights.  
This makes it easier to try different models and build on their capabilities.

### Main Concepts

 - Model  
   The neural network. It maps prepared inputs to outputs such as embeddings, scores or generated text.
 - Checkpoint  
   A saved configuration and learned weights.  
   It specifies a particular version of a model.
 - Tokenizer  
   Converts text into token IDs. The model then maps these IDs to learned embeddings.
 - Processor  
   Prepares inputs in the format expected by the model.  
   For image and text models, it combines image preprocessing with tokenization.
 - Auto Classes:  
   Classes such as `AutoModel`, `AutoTokenizer` and `AutoProcessor`.  
   They select the appropriate implementation for a given checkpoint.
 - Pipeline  
   A high level interface for supported tasks.  
   It combines preprocessing, model inference and output processing.

### The Hugging Face Hub

The [Hugging Face Hub](https://huggingface.co/models) hosts model repositories.  
A repository may contain weights, configuration files, preprocessing settings and documentation.  
A model is usually identified by an `organization/model-name` string.

The method `from_pretrained()` loads the relevant files from a repository or a local folder.  
Downloaded files are cached for reuse. The model can then run locally on a CPU or a GPU.

### In This Notebook

The notebooks demonstrates loading a pre trained image and text embedding model and its matching processor.  
The processor prepares the images and text. The model produces their embeddings.  
Then it compare the embeddings to classify images without training a new classifier.

* <font color='red'>(**?**)</font> Without the `transformers` package, how would you push a string of text into a Pytorch model?
* <font color='brown'>(**#**)</font> Each processor is associated with a specific checkpoint as the model.  
  Different checkpoints may require different tokenization and image preprocessing.

## The SigLIP Model

The CLIP model has few weaknesses:
 - Coupled Comparisons  
   The probability of one pair depends on the other candidates in its row or column.  
   The loss cannot evaluate each pair independently.
 - Global Normalization  
   Softmax needs normalization over all candidates in a row or column.  
   This complicates memory efficient training across multiple devices.
 - Small Batch Performance  
   CLIP's Softmax loss benefits from many mismatched pairs in a batch.  
   It can be less effective when only small batches are practical.

The _SigLIP_ models are evolution of the _CLIP_ model.
SigLIP keeps the same architecture of an image encoder and a text encoder.  
The main change is the _Loss Function_.  
It replaces the loss over a batch of candidates into a per pair loss: **Find the best match from the candidates** → **Score each pair match**.

 - Independent Pair Loss  
   Each image and text pair becomes a binary classification task: _Match / No Match_.  
   Its cosine similarity is scaled and shifted by learned parameters.  
   A _Sigmoid_ converts this score into a value between 0 and 1.
 - No Global Normalization  
   Each pair contributes its own loss. No row or column normalization is needed.  
   Pair losses can be computed in chunks and combined.  
   This enables more memory efficient training across multiple devices.
 - Better Small Batch Performance  
   The [SigLIP paper](https://arxiv.org/abs/2303.15343) reports better results at smaller batch sizes than the Softmax loss.  
   The method also scales to large batches. Large batches can still be useful.

For matching training pairs, the target is **1**. For mismatched pairs, it is **0**.  
The _Binary Cross Entropy Loss_ penalizes incorrect predictions for each pair.  
The pair losses are combined to update both encoders.

* <font color='brown'>(**#**)</font> The Sigmoid scores do not need to sum to 1 across a row or column.  
  One pair's score does not need to decrease just because another pair receives a higher score.
* <font color='brown'>(**#**)</font> SigLIP still treats off diagonal training pairs as mismatches.  
  Removing Softmax does not solve the problem of related pairs being labeled as negatives.
* <font color='brown'>(**#**)</font> The _SigLIP 2_ model mainly focuses on improving the training process: Masked patch prediction, Region level supervision, Multi language, Flexible resolution, etc..

In [ ]:
# Model

oModel     = AutoModel.from_pretrained(modelId).to(runDevice).eval()
oProcessor = AutoProcessor.from_pretrained(modelId)

In [ ]:
# Load Image

imgIdx = random.randint(0, len(lImgFile) - 1)
imgFileName = lImgFile[imgIdx]
imgFilePath = os.path.join(dataDirPath, imgFileName)
tI = iio.imread(imgFilePath)

hF, hA = plt.subplots(figsize = (8, 6))
hA.imshow(tI)
hA.set_title(f'File: {imgFileName}')
hA.axis('off');

In [ ]:
# Embedding Example
# This shows a joint call of both image and text.

imgCls = dData[imgFileName]
sampleStr = f'This is a photo of a {imgCls}'.lower()

dInputs = oProcessor(images = tI, text = sampleStr, return_tensors = 'pt',
                     padding = 'max_length', truncation = True,
                     max_length = oModel.config.text_config.max_position_embeddings).to(runDevice)

with torch.inference_mode():
    oOutput = oModel(**dInputs, return_dict = True)

# Embedding of the image.
vImgEmb = oOutput.image_embeds[0].cpu().numpy() #<! Or `oOutput['image_embeds']`

# Embedding of the text.
vTxtEmb = oOutput.text_embeds[0].cpu().numpy() #<! Or `oOutput['text_embeds']`

print(f'Text: {sampleStr}')
print(f'Image embedding shape: {vImgEmb.shape}')
print(f'Text embedding shape: {vTxtEmb.shape}')
print(f'Cosine similarity: {np.dot(vImgEmb, vTxtEmb)}') 
# The scaled similarity in SigLIP (Logit Similarity): exp(logitScale) * cosineSim + logitBias.
# The values of `logitScale` and `logitBias` are learned by the model.
# Calculated since both data types are on the same call.
print(f'Scaled similarity (Logit): {oOutput.logits_per_image[0, 0].item():0.3f}') 

In [ ]:
# Embedding Example
# This shows a separate call for image and text.
# Each encoder runs independently. Normalize the vectors to match the joint call.
# Similarity is calculated from the two vectors below.

imgCls = dData[imgFileName]
sampleStr = f'This is a photo of a {imgCls}'.lower()

# Embedding of the image.
dImgInputs = oProcessor(images = tI, return_tensors = 'pt').to(runDevice)
with torch.inference_mode():
    oImgOutput = oModel.get_image_features(**dImgInputs, return_dict = True)

vImgEmb = oImgOutput.pooler_output[0].cpu().numpy()
vImgEmb = vImgEmb / np.linalg.norm(vImgEmb)

# Embedding of the text.
dTxtInputs = oProcessor(text = sampleStr, return_tensors = 'pt',
                        padding = 'max_length', truncation = True,
                        max_length = oModel.config.text_config.max_position_embeddings).to(runDevice)
with torch.inference_mode():
    oTxtOutput = oModel.get_text_features(**dTxtInputs, return_dict = True)

vTxtEmb = oTxtOutput.pooler_output[0].cpu().numpy()
vTxtEmb = vTxtEmb / np.linalg.norm(vTxtEmb)

cosineSim = np.dot(vImgEmb, vTxtEmb)
logitSim  = oModel.logit_scale.exp().item() * cosineSim + oModel.logit_bias.item()

print(f'Text: {sampleStr}')
print(f'Image embedding shape: {vImgEmb.shape}')
print(f'Text embedding shape: {vTxtEmb.shape}')
print(f'Cosine similarity: {cosineSim}')
print(f'Scaled similarity (Logit): {logitSim:0.3f}')

* <font color='red'>(**?**)</font> Does low similarity score means no match? Think about the use case.

## Zero Shot Image Classification

### Zero Shot Classification

_Zero Shot Classification_ assigns inputs to classes without training on labeled examples for the target task.  
The model reuses knowledge learned during _Training_ for a new task.  
We describe the candidate classes instead of training a new classifier.

* <font color='brown'>(**#**)</font> Zero shot does not mean the model was never trained.  
  It may have encountered these concepts during pre training. No task specific training is performed here.

### Zero Shot Image Classification with Joint Image and Text Embedding Space

The shared space lets us compare an image with descriptions of the candidate classes.  
Each description acts as a reference point for its class.

1. Write a description for **each class**.  
   For example, _"This is a photo of a {candidate class}"_.
2. Encode the descriptions with the _Text Encoder_.
3. Encode the input image with the _Image Encoder_.
4. Normalize the embeddings and calculate their _Cosine Similarity_.
5. Predict the class whose description has the highest, among candidates, similarity to the image.  

For our dataset, each image is compared with **all 10 class descriptions**.  
Its true label is used only to evaluate the prediction.  
The text embeddings can be calculated once and reused for all images.

Mathematically the operation is equivalent to:

$$ \hat{y}_{i} = \arg \max_{j \in \{1, \ldots, C\}} {s}_{ij}, \qquad {s}_{ij} = \hat{\boldsymbol{t}}_{j}^{T} \hat{\boldsymbol{v}}_{i} $$

Where:

 - ${I}_{i}$ is the input image.
 - ${T}_{j}$ is the text description of candidate class $j$.
 - $C$ is the number of candidate classes.
 - $\boldsymbol{v}_{i} = {f}_{\boldsymbol{\theta}} \left( {I}_{i} \right)$ is the image embedding.
 - $\hat{\boldsymbol{v}}_{i} = \frac{\boldsymbol{v}_{i}}{\left\| \boldsymbol{v}_{i} \right\|_{2}}$ is the normalized image embedding.
 - ${f}_{\boldsymbol{\theta}} \left( \cdot \right)$ is the _image encoder_ with learned parameters $\boldsymbol{\theta}$.
 - $\boldsymbol{t}_{j} = {g}_{\boldsymbol{\phi}} \left( {T}_{j} \right)$ is the text embedding.
 - $\hat{\boldsymbol{t}}_{j} = \frac{\boldsymbol{t}_{j}}{\left\| \boldsymbol{t}_{j} \right\|_{2}}$ is the normalized text embedding.
 - ${g}_{\boldsymbol{\phi}} \left( \cdot \right)$ is the _text encoder_ with learned parameters $\boldsymbol{\phi}$.
 - ${s}_{ij}$ is the cosine similarity between image $i$ and the description of class $j$.
 - $\hat{y}_{i}$ is the predicted class index for image $i$.

* <font color='brown'>(**#**)</font> The winning similarity does not need to be close to 1.  
  Its ranking among the candidates determines the prediction.  
* <font color='red'>(**?**)</font> What happens if the image belongs to a class missing from the candidate list?
* <font color='brown'>(**#**)</font> One may want to enforce minimal threshold for a decision.
* <font color='brown'>(**#**)</font> The wording of a description affects its embedding and may change the prediction.

In [ ]:
# Class Prediction for the Image Text Embedding Model
# Estimate the class of the image.

lClsText = [f'This is a photo of a {clsName}'.lower() for clsName in lClsName]

dTxtInputs = oProcessor(text = lClsText, return_tensors = 'pt',
                        padding = 'max_length', truncation = True,
                        max_length = oModel.config.text_config.max_position_embeddings).to(runDevice)
with torch.inference_mode():
    oTxtOutput = oModel.get_text_features(**dTxtInputs, return_dict = True)

mTxtEmb = oTxtOutput.pooler_output.cpu().numpy()
mTxtEmb = mTxtEmb / np.linalg.norm(mTxtEmb, axis = 1, keepdims = True)

dImgInputs = oProcessor(images = tI, return_tensors = 'pt').to(runDevice)
with torch.inference_mode():
    oImgOutput = oModel.get_image_features(**dImgInputs, return_dict = True)

vImgEmb = oImgOutput.pooler_output[0].cpu().numpy()
vImgEmb = vImgEmb / np.linalg.norm(vImgEmb)

vSimilarity = mTxtEmb @ vImgEmb
predClsIdx  = np.argmax(vSimilarity)
predClsName = lClsName[predClsIdx]

print(f'Predicted class: {predClsName}')
print(f'Cosine similarity: {vSimilarity[predClsIdx]:0.3f}')

dfScores = pd.DataFrame({'Class': lClsName, 'Cosine Similarity': vSimilarity})
display(dfScores.sort_values('Cosine Similarity', ascending = False).reset_index(drop = True))

* <font color='blue'>(**!**)</font> Run the above without the `.lower()` method.  
  What can you say about the training of the model?

### Processing Batch of Images

Let $N$ be the number of images, $C$ the number of candidate classes and $D$ the embedding dimension.  
Stack the normalized embeddings as columns:

$$ \boldsymbol{T} = \begin{bmatrix} \hat{\boldsymbol{t}}_{1} & \cdots & \hat{\boldsymbol{t}}_{C} \end{bmatrix} \in \mathbb{R}^{D \times C}, \qquad \boldsymbol{V} = \begin{bmatrix} \hat{\boldsymbol{v}}_{1} & \cdots & \hat{\boldsymbol{v}}_{N} \end{bmatrix} \in \mathbb{R}^{D \times N} $$

Each column has unit L2 norm. One matrix multiplication gives all cosine similarities:

$$ \boldsymbol{S} = \boldsymbol{T}^{T} \boldsymbol{V} \in \mathbb{R}^{C \times N}, \qquad S_{ji} = \hat{\boldsymbol{t}}_{j}^{T} \hat{\boldsymbol{v}}_{i} $$

 - Row $j$ contains the scores of class $j$ for all images.
 - Column $i$ contains the scores of all classes for image $i$.

Choose the largest score in each column:

$$ \hat{y}_{i} = \arg \max_{j \in \{1, \ldots, C\}} S_{ji} $$

The code stores embeddings as **rows** in `mTxtEmb` and `mImgEmb`.  
Hence, `mS = mTxtEmb @ mImgEmb.T` and `vYPred = np.argmax(mS, axis = 0)`.

The text embeddings are calculated once. Images are encoded in small batches to limit memory usage.  
Image embeddings and predictions follow the order of `lImgFile`. Labels are used only for evaluation.

In [ ]:
# Dataset Analysis

batchSize  = 32
numSamples = len(lImgFile)
lClsText   = [f'This is a photo of a {clsName}'.lower() for clsName in lClsName]

dTxtInputs = oProcessor(text = lClsText, return_tensors = 'pt',
                        padding = 'max_length', truncation = True,
                        max_length = oModel.config.text_config.max_position_embeddings).to(runDevice)
with torch.inference_mode():
    oTxtOutput = oModel.get_text_features(**dTxtInputs, return_dict = True)

mTxtEmb = oTxtOutput.pooler_output.cpu().numpy()
mTxtEmb = mTxtEmb / np.linalg.norm(mTxtEmb, axis = 1, keepdims = True)

mImgEmb = np.empty((numSamples, mTxtEmb.shape[1]), dtype = mTxtEmb.dtype)

for batchStart in range(0, numSamples, batchSize):
    batchEnd  = min(batchStart + batchSize, numSamples)
    lBatchImg = [iio.imread(os.path.join(dataDirPath, imgFile)) for imgFile in lImgFile[batchStart:batchEnd]]
    dImgInputs = oProcessor(images = lBatchImg, return_tensors = 'pt').to(runDevice)

    with torch.inference_mode():
        oImgOutput = oModel.get_image_features(**dImgInputs, return_dict = True)

    mImgEmb[batchStart:batchEnd, :] = oImgOutput.pooler_output.cpu().numpy()

mImgEmb = mImgEmb / np.linalg.norm(mImgEmb, axis = 1, keepdims = True)
mS      = mTxtEmb @ mImgEmb.T
vYPred  = np.argmax(mS, axis = 0)
vY      = np.array([lClsName.index(dData[imgFileName]) for imgFileName in lImgFile])

print(f'Score matrix shape (Classes, Images): {mS.shape}')
print(f'Number of predictions: {len(vYPred)}')

In [ ]:
# Display Results

dScore = {
    'Accuracy': accuracy_score(vY, vYPred),
    'Macro F1': f1_score(vY, vYPred, average = 'macro', zero_division = 0),
}

for scoreName, scoreVal in dScore.items():
    print(f'{scoreName}: {scoreVal:0.2%}')

hF, hA = plt.subplots(figsize = (8, 7))
hA, mConfMat = PlotConfusionMatrix(vY, vYPred, normMethod = 'true', hA = hA,
                                  lLabels = lClsName, dScore = dScore,
                                  xLabelRot = 45, valFormat = '0.1%')
hF.tight_layout()

* <font color='green'>(**@**)</font> Redo the task with `CLIP` model.

## The Embedding Space

The images and class descriptions are points in a shared, high dimensional space.  
_Dimensionality Reduction_ lets us inspect some of its structure in 2D.

We apply _UMAP_ to both sets **together**, using _Cosine Distance_.  
The method aims to preserve local neighborhoods, not every distance.

 - Circles represent images.
 - Stars represent the 10 class descriptions.
 - Color identifies the class. Labels are used only for coloring, not for fitting UMAP.

* <font color='brown'>(**#**)</font> The 2D layout is an approximation. Distances and class separation can be distorted.  
  Classification still uses similarities in the original embedding space.
* <font color='red'>(**?**)</font> Do images of the same class group together? Where is their text embedding?

In [ ]:
# Visualization by Dimensionality Reduction

numNeighbors = 15
minDist      = 0.1

mX = np.vstack((mImgEmb, mTxtEmb))
oDimRed = UMAP(n_components = 2, n_neighbors = numNeighbors, min_dist = minDist,
               metric = 'cosine', random_state = seedNum, n_jobs = 1)
mZ = oDimRed.fit_transform(mX)

mZImg = mZ[:mImgEmb.shape[0], :]
mZTxt = mZ[mImgEmb.shape[0]:, :]

hF, hA = plt.subplots(figsize = (8, 5))

for clsIdx, clsName in enumerate(lClsName):
    vClsMask = vY == clsIdx
    clsColor = lMatPltLibclr[clsIdx % len(lMatPltLibclr)]

    hA.scatter(mZImg[vClsMask, 0], mZImg[vClsMask, 1], color = clsColor,
               marker = 'o', s = 35, alpha = 0.65, label = clsName)
    hA.scatter(mZTxt[clsIdx, 0], mZTxt[clsIdx, 1], color = clsColor,
               marker = '*', s = 220, edgecolors = 'k', linewidths = 0.8, zorder = 3)

hClassLegend = hA.legend(title = 'Class', loc = 'upper left', bbox_to_anchor = (1.02, 1))
hA.add_artist(hClassLegend)

hImgMarker = hA.scatter([], [], color = 'gray', marker = 'o', s = 35, label = 'Image')
hTxtMarker = hA.scatter([], [], color = 'gray', marker = '*', s = 220,
                       edgecolors = 'k', linewidths = 0.8, label = 'Class Description')
hA.legend(handles = [hImgMarker, hTxtMarker], title = 'Modality',
          loc = 'lower left', bbox_to_anchor = (1.02, 0))

hA.set_title('UMAP: Image and Text Embeddings')
hA.set_xlabel('UMAP 1')
hA.set_ylabel('UMAP 2')
hA.set_aspect('equal', adjustable = 'datalim')
hF.tight_layout()

* <font color='blue'>(**!**)</font> Run analysis of Text to Text similarities and Image to Image (Within the same class) similarities.
* <font color='red'>(**?**)</font> Explain the embedding space. How come text is far from all yet results are great?